## Cellule 1 — Imports et chargement des données

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import random

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Copies des labels entiers pour l'affichage final
y_train_labels = y_train.copy()
y_test_labels = y_test.copy()

print("x_train shape :", x_train.shape)
print("y_train shape :", y_train.shape)
print("x_test shape  :", x_test.shape)
print("y_test shape  :", y_test.shape)

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()


## Partie 1 — Prétraitement des données 

In [ ]:

x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test,  10)

# Vérifications
print('Min x_train :', x_train.min())
print('Max x_train :', x_train.max())
print('Forme de y_train après encodage :', y_train.shape)


## Partie 1 — Construction du CNN


In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),

    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
])

model.summary()


## Partie 1 — Compilation et entraînement

In [ ]:

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

## Partie 1 — Visualisation des courbes

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.title("Évolution de l'accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title("Évolution de la loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## Partie 1 — Évaluation et prédiction 

In [ ]:

loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f'Accuracy test : {accuracy * 100:.2f}%')


index = random.randint(0, len(x_test) - 1)

prediction = model.predict(np.expand_dims(x_test[index], axis=0), verbose=0)
pred_label = int(np.argmax(prediction))
true_label = int(y_test_labels[index][0])

print('Classe réelle  :', class_names[true_label])
print('Classe prédite :', class_names[pred_label])

plt.figure(figsize=(4, 4))
plt.imshow(x_test[index])
plt.title(f'Vrai : {class_names[true_label]}\nPrédit : {class_names[pred_label]}')
plt.axis('off')
plt.show()


In [ ]:
## Partie 2 — Petite modification expérimentale

In [ ]:

model_sgd = keras.models.Sequential([
    keras.layers.Input(shape=(32, 32, 3)),

    keras.layers.Conv2D(32, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
])

model_sgd.compile(
    optimizer='sgd',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# TODO 10 : entraîner le modèle avec SGD
history_sgd = model_sgd.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

# TODO 11 : évaluer sur le test
loss_sgd, accuracy_sgd = model_sgd.evaluate(x_test, y_test, verbose=0)

print('Accuracy modèle de base :', round(accuracy * 100, 2), '%')
print('Accuracy modèle avec SGD :', round(accuracy_sgd * 100, 2), '%')

Q1: Un CNN exploite la structure spatiale des images grâce à des filtres convolutifs qui détectent des motifs locaux (bords, textures) quelle que soit leur position dans l'image. Contrairement à un réseau Dense qui traite chaque pixel indépendamment, le CNN partage ses poids entre toutes les positions (partage de paramètres), ce qui le rend bien plus efficace et robuste aux translations.

Q2 : réduit la dimension spatiale des feature maps en conservant uniquement la valeur maximale dans chaque fenêtre (ex. 2×2). Cela diminue le nombre de paramètres, accélère le calcul, et apporte une certaine invariance aux petites translations, tout en conservant les caractéristiques les plus importantes.

Q3: Le modèle avec SGD obtient donc une accuracy plus faible sur le test. Avec plus d'epochs et un bon learning rate

Q4 : Un signe typique d'overfitting est une accuracy d'entraînement qui continue d'augmente pendant que l'accuracy de validation stagne ou diminue (et inversement, la loss de validation remonte alors que la loss d'entraînement baisse). Cela indique que le modèle mémorise les données d'entraînement sans généraliser.